# Türk Plaka Tespiti — YOLOv8 Eğitimi (Google Colab)

Bu notebook, Türk araç plakalarının **konumunu** tespit eden bir YOLOv8n modelini ücretsiz Colab GPU'su ile eğitir. (OCR / karakter okuma bu aşamanın kapsamı dışındadır — Aşama 2'de gelecek.)

**Başlamadan önce:**
1. Menüden **Runtime → Change runtime type → T4 GPU** seçin.
2. Roboflow API anahtarınızı hazırlayın: https://app.roboflow.com/settings/api
   - Önerilen: Colab'ın sol panelindeki 🔑 **Secrets** bölümüne `ROBOFLOW_API_KEY` adıyla ekleyin.
   - Eklemezseniz aşağıdaki hücre anahtarı güvenli şekilde (gizli girişle) soracaktır.

Veri seti: [License Plates of Vehicles in Turkey](https://universe.roboflow.com/tr-plaka-recognition/license-plates-of-vehicles-in-turkey-s3tbj-s5lcc) — 3.501 görüntü, tek sınıf, **CC BY 4.0**.

In [ ]:
# 1) Kurulum
%pip -q install ultralytics roboflow

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK — Runtime > Change runtime type'tan GPU seçin!")

In [ ]:
# 2) Roboflow API anahtarı (koda gömmeyin!)
import os

if not os.environ.get("ROBOFLOW_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
        print("API anahtarı Colab Secrets'tan alındı.")
    except Exception:
        from getpass import getpass
        os.environ["ROBOFLOW_API_KEY"] = getpass("Roboflow API anahtarınızı girin: ")

In [ ]:
# 3) Veri setini indir (YOLOv8 formatında)
from pathlib import Path
from roboflow import Roboflow

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("tr-plaka-recognition").project("license-plates-of-vehicles-in-turkey-s3tbj-s5lcc")
latest = max(int(v.version.split("/")[-1]) for v in project.versions())
dataset = project.version(latest).download("yolov8", location="datasets/turkish-plates")

DATA_YAML = Path(dataset.location) / "data.yaml"

# data.yaml'daki göreli yolları indirilen konuma göre mutlaklaştır
# (Roboflow export'u bazen "../train/images" gibi yollar içerir)
import yaml
cfg = yaml.safe_load(DATA_YAML.read_text())
for split, folder in (("train", "train"), ("val", "valid"), ("test", "test")):
    d = Path(dataset.location) / folder / "images"
    if split in cfg and d.exists():
        cfg[split] = str(d.resolve())
DATA_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False))

print("data.yaml:", DATA_YAML)

In [ ]:
# 4) Eğitim — yolov8n, 50 epoch, 640px, batch 16
# Not: GPU belleği yetmezse (CUDA out of memory) batch'i 8'e düşürün.
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=16,
    project="runs",
    name="plate-detect",
)

In [ ]:
# 5) En iyi ağırlığı belirgin bir yere kopyala (yeniden bağlanmaya dayanıklı)
from pathlib import Path
import shutil
from ultralytics import YOLO

# Eğitim çıktısındaki en iyi ağırlık diskte durur; en yeni çalıştırmayı bul
candidates = sorted(Path("runs").glob("plate-detect*/weights/best.pt"),
                    key=lambda p: p.stat().st_mtime)
if not candidates:
    raise SystemExit("best.pt bulunamadı — 4. hücredeki eğitim tamamlanmamış olabilir. "
                     "Önce 4. hücreyi çalıştırıp bitmesini bekleyin.")
best_path = candidates[-1]

Path("weights").mkdir(exist_ok=True)
shutil.copy2(best_path, "weights/best.pt")
print("En iyi ağırlık:", best_path, "→ weights/best.pt")

best = YOLO("weights/best.pt")

In [ ]:
# 6) Değerlendirme — val seti metrikleri
metrics = best.val(data=str(DATA_YAML), split="val")

rows = {
    "mAP@0.5": metrics.box.map50,
    "mAP@0.5:0.95": metrics.box.map,
    "Precision": metrics.box.mp,
    "Recall": metrics.box.mr,
}

print("\n| Metrik | Değer |")
print("|---|---|")
for name, value in rows.items():
    print(f"| {name} | {value:.4f} |")
print("\nBu tabloyu README'nin 'Değerlendirme Sonuçları' bölümüne yapıştırın.")

In [ ]:
# 7) Örnek val görüntülerinde tahmin kutularını çiz ve results/ altına kaydet
import random

val_dir = DATA_YAML.parent / "valid" / "images"
images = sorted(val_dir.glob("*.jpg")) + sorted(val_dir.glob("*.png"))
sample = random.sample(images, min(8, len(images)))

best.predict(source=[str(p) for p in sample], save=True,
             project="results", name="val_predictions", exist_ok=True)

from IPython.display import Image as IPyImage, display
for p in sorted(Path("results/val_predictions").glob("*"))[:4]:
    display(IPyImage(filename=str(p), width=480))

In [ ]:
# 8) best.pt ve örnek çıktıları bilgisayarınıza indirin
# (İndirdiğiniz best.pt'yi yerel repoda weights/ klasörüne koyarsanız
#  predict.py doğrudan çalışır.)
shutil.make_archive("val_predictions", "zip", "results/val_predictions")

from google.colab import files
files.download("weights/best.pt")
files.download("val_predictions.zip")

## Sonraki adım

İndirdiğiniz `best.pt` ile yerel makinede çıkarım yapın:

```bash
python predict.py --source ornek.jpg --conf 0.25
```

Kırpılmış plaka görüntüleri `predictions/crops/` altına kaydedilir — bunlar **Aşama 2 (OCR)** girdileri olacak.